# GR00T N1.7 학습 방식 이해하기 — LLM/VLM SFT 경험자를 위한 안내

> **이 노트북은 실행용이 아니라 *설명용*** 입니다. 셀은 개념을 보여주는 예시 코드이고,
> 실제 GR00T 의존성(`gr00t`, `lerobot`)이나 GPU 없이 읽기만 해도 이해되도록 썼습니다.
>
> 출처: NVIDIA [Isaac-GR00T 레포](https://github.com/NVIDIA/Isaac-GR00T),
> [GR00T-N1.7-3B 모델카드](https://huggingface.co/nvidia/GR00T-N1.7-3B), [백서](https://arxiv.org/abs/2503.14734).
> (필드명·경로는 버전에 따라 조금씩 달라질 수 있으니 최종은 레포로 확인)

## 당신의 출발점

당신이 해본 것:
1. HF에서 모델 다운로드
2. 학습 데이터를 **그 모델의 chat 템플릿**으로 변환 (`{"messages":[...]}` -> 토큰 시퀀스)
3. **SFT 라이브러리**(trl 등)로 next-token 예측 파인튜닝

GR00T도 "HF에서 받아 -> 데이터를 *그 모델이 먹는 형식*으로 변환 -> 파인튜닝"이라는 **뼈대는 똑같아요.**
다른 건 딱 두 군데입니다:
- **데이터의 형식** — 채팅 텍스트가 아니라 **시간 정렬된 (영상 + 상태 + 행동) 에피소드**
- **예측 대상** — 다음 토큰이 아니라 **다음 행동 청크(연속값)를 디퓨전으로 회귀**

이 노트북은 그 두 차이를 당신 경험에 매핑해서 풀어줍니다.

## 1. 큰 그림: LLM SFT vs VLA(GR00T) 한눈 비교

| | LLM/VLM SFT (당신이 해본 것) | GR00T N1.7 (VLA) |
|---|---|---|
| 한 **샘플**의 정체 | 대화 한 건 (`messages`) | **시간 창(window)**: 과거 프레임+상태 -> 미래 행동 청크 |
| 입력 | 텍스트(+이미지) 토큰 | **영상 프레임 + proprioception 상태 벡터 + 언어 + embodiment ID** |
| 출력/정답 | 다음 **토큰** (이산, 교차엔트로피) | **행동 벡터 청크** (연속, 디퓨전 denoising) |
| 데이터 형식 | JSONL of chat | **LeRobot v2 디렉토리** (parquet + mp4 + meta) |
| "템플릿" 역할 | chat 템플릿 | **`modality.json` + embodiment tag** (어느 컬럼이 상태/행동/카메라인지) |
| 정규화 | 토크나이저가 처리 | **상태/행동을 통계로 정규화**(평균/분위수) — 명시적 단계 |
| 파인튜닝 도구 | `trl` SFTTrainer 등 | `gr00t/experiment/launch_finetune.py` |

핵심 직관: **LLM은 "텍스트를 토큰으로 쪼개" 학습하고, VLA는 "로봇 경험을 시간 스텝으로 쪼개" 학습한다.**

## 2. GR00T가 먹는 것 / 뱉는 것

모델카드 기준 입력 4종 -> 출력 1종:

```
입력:
  (1) Vision        : uint8 RGB 프레임 여러 장 (카메라들)
  (2) State         : float proprioception 벡터 (관절각/EE pose 등 "지금 내 몸 상태")
  (3) Language      : 지시문 문자열  ("pick up the red cube")
  (4) Embodiment ID : 정수 (어떤 로봇인지 — 상태/행동 키와 정규화를 고름)
출력:
  (5) Action chunk  : 연속값 행동 벡터들 (미래 H 스텝) — 디퓨전 DiT로 한 번에 생성
```

아키텍처는 **2-시스템**: VLM(Cosmos-Reason2-2B)이 영상+언어로 추론 -> 그 위에 **32층 DiT 디퓨전**이
현재 상태와 합쳐 정밀한 행동으로 denoise. 당신이 앞서 말한 "별도 상태 입력 + 디퓨전으로 궤적 한 번에"가 정확히 이거예요.

## 3. 디스크 위 데이터셋 구조 (LeRobot v2 + modality.json)

이게 당신의 `train.jsonl`에 해당하는 것 — 단, 파일 하나가 아니라 **폴더 트리**입니다:

```
my_dataset/
|-- meta/
|   |-- info.json        # 데이터셋 메타 (fps, 피처 스키마, 버전)
|   |-- episodes.jsonl   # 에피소드별 인덱스/길이
|   |-- tasks.jsonl      # 언어 지시문들 (task index -> 문장)
|   |-- modality.json    # * GR00T 전용: 어떤 컬럼이 상태/행동/카메라인지 매핑
|   `-- stats.json       # 상태/행동 정규화 통계 (평균/표준편차/분위수)
|-- data/
|   `-- chunk-000/
|       |-- episode_000000.parquet   # 스텝마다 한 행: state, action, timestamp, ...
|       `-- episode_000001.parquet
`-- videos/
    `-- chunk-000/
        `-- observation.images.base_camera/
            |-- episode_000000.mp4    # 그 에피소드 카메라 영상
            `-- episode_000001.mp4
```

- **parquet** = 수치(상태/행동/타임스탬프). 픽셀은 안 넣음.
- **mp4** = 영상. parquet의 타임스탬프와 **시간 정렬**되어, 학습 시 로더가 "이 스텝의 프레임"을 뽑아옴.
- **meta/** = 스키마/정규화/언어. 모델이 데이터를 해석하는 "사용설명서".

> LLM 비유: parquet+mp4 = 본문 데이터, `meta/`의 `modality.json`+`stats.json` = chat 템플릿+토크나이저 설정에 해당.

In [ ]:
# 한 에피소드 parquet의 '한 행(=한 타임스텝)'은 개념적으로 이렇게 생겼다 (예시):
example_row = {
    "timestamp":         0.05,                  # 이 스텝 시각 (fps로 정렬)
    "observation.state": [0.01, -0.20, ...],    # (2) proprioception 벡터 (예: 관절 9 or EE pose)
    "action":            [0.004, -0.003, ...],  # (5) 이 스텝의 행동 (정답 라벨)
    "task_index":        2,                      # tasks.jsonl 의 어떤 지시문인지
    "episode_index":     0,
    "frame_index":       0,
    # 픽셀은 여기 없음 -> videos/.../episode_000000.mp4 의 같은 timestamp 프레임으로 로더가 채움
}
# LLM SFT의 한 행이 {"messages":[...]} 였다면, VLA의 한 행은 '한 시점의 (상태, 행동)' 이다.

## 4. `modality.json` — VLA의 "chat 템플릿"에 해당하는 핵심 파일

parquet의 `observation.state`/`action`은 그냥 **숫자 배열**이에요. 이 배열의 *어디부터 어디까지가 무슨 의미인지*를
`modality.json`이 알려줍니다. 즉 "긴 벡터를 명명된 필드로 쪼개는 지도".

예시 (단일 팔 + 그리퍼라고 가정):

In [ ]:
modality_json = {
  "state": {
    # observation.state 벡터를 이름 붙은 구간으로 분해
    "arm":     {"start": 0, "end": 7},   # 관절 7개
    "gripper": {"start": 7, "end": 8}    # 그리퍼 1
  },
  "action": {
    "arm":     {"start": 0, "end": 7},
    "gripper": {"start": 7, "end": 8}
  },
  "video": {
    # 어떤 카메라 키가 있는지 (videos/ 폴더 이름과 대응)
    "base_camera": {"original_key": "observation.images.base_camera"}
  },
  "annotation": {
    "human.task_description": {}   # 언어 지시문 출처
  }
}
# 이게 있어야 모델이 "이 8차원 중 0~7은 팔, 7~8은 그리퍼" 라고 해석하고,
# embodiment 별 정규화/행동공간을 올바르게 적용한다.

## 5. Embodiment Tag — "어떤 로봇이냐"

GR00T는 여러 로봇을 한 모델로 다루므로, **embodiment tag**로 "지금 이 데이터는 무슨 로봇"인지 지정합니다.
이 태그가 *어떤 modality 설정(상태/행동 키, 정규화 통계)* 을 쓸지 고릅니다.

- 사전등록 예: `LIBERO_PANDA` (Franka Panda — LIBERO sim), `OXE_DROID_...` (DROID 하드웨어)
- 새 로봇/새 행동공간이면: `NEW_EMBODIMENT` 로 등록하고 `modality.json` + 통계를 직접 제공

> **우리 ManiSkill 데이터는 Franka Panda** 라서 `LIBERO_PANDA`에 가깝지만,
> 행동공간이 다르면(아래 6/8절) `NEW_EMBODIMENT`로 등록하는 편이 안전.

## 6. Action Horizon(청킹)과 행동 공간

**Action horizon** = 한 번에 예측하는 미래 행동 스텝 수 (예: `--action-horizon 16`).
디퓨전 헤드가 "다음 1스텝"이 아니라 **다음 16스텝의 행동 궤적을 통째로** 생성 -> 부드럽고, 추론 횟수가 1/16로 줄어듦.

> LLM 비유: 토큰 1개씩 자기회귀로 뽑는 대신, **문장 16토큰을 한 번에 비자기회귀로 뱉는** 느낌.

**행동 공간**: N1.7 기본은 **상대 EE(end-effector) 행동 공간**(현재 pose 대비 델타)을 인간/로봇 공유로 씀.
우리 ManiSkill 데이터는 **관절 위치(`pd_joint_pos`, 8차원)** 라 표현이 다름 -> 두 갈래:
- 관절 그대로 두고 `NEW_EMBODIMENT`로 등록(관절 행동공간), 또는
- 기록된 `tcp_pose`로 **EE 델타를 사후 계산**해 N1.7 관례에 맞춤 (앞 대화에서 본 방법).

## 7. "한 학습 샘플" = 시간 창 (LLM의 'chat 한 건'에 대응)

로더가 디스크에서 만들어 모델에 넣는 **배치 1개 요소**는 대략 이렇게 생겼습니다.
이게 당신의 "토큰화된 chat 한 건"에 해당하는 VLA 버전이에요:

In [ ]:
# 개념적 — 실제 텐서 키/shape는 data config(transform)가 결정
training_sample = {
    # (1) 영상: (카메라당) 과거 몇 프레임  [T_obs, H, W, 3] uint8
    "video.base_camera": "<uint8 frames around timestep t>",

    # (2) 상태: 현재(및 과거) proprioception  [T_obs, state_dim] float
    "state.arm":     "<joint angles>",
    "state.gripper": "<gripper opening>",

    # (3) 언어: tasks.jsonl 에서 가져온 지시문
    "annotation.human.task_description": "pick up the red cube",

    # (4) embodiment: 어떤 로봇인지 (정규화/키 선택)
    "embodiment_id": "LIBERO_PANDA",   # 또는 NEW_EMBODIMENT

    # (5) 정답 = 미래 H스텝 행동 청크  [H, action_dim] float  <- 디퓨전이 이걸 맞추도록 학습
    "action.arm":     "<future arm actions, horizon H>",
    "action.gripper": "<future gripper actions, horizon H>",
}
# 학습 손실: 다음 토큰 CE 가 아니라, 이 action 청크에 대한 디퓨전(flow) denoising 손실.

## 8. 우리 ManiSkill 데이터 -> GR00T(LeRobot) 변환 매핑

우리가 이미 만든 `data/datasets/ThreeColoredCubes-v1/*.rgb.*.h5` 안의 내용이 어디로 가는지:

| 우리 h5 안의 것 | GR00T/LeRobot 어디로 | 비고 |
|---|---|---|
| `obs/sensor_data/base_camera/rgb` (T,128,128,3) | `videos/.../observation.images.base_camera/episode_*.mp4` | 128px -> 모델 기대 해상도로 리사이즈 |
| `obs/agent/qpos` (T,9) 또는 `obs/extra/tcp_pose` | parquet `observation.state` (+ `modality.json` state 매핑) | proprioception 선택 |
| `actions` (T-1,8) (관절) | parquet `action` | EE델타로 변환할지 결정 (6절) |
| `target_id` + `.json`의 `label_metadata` | `tasks.jsonl` 지시문 ("pick up the {color} cube") | 우리가 만든 라벨 메타데이터가 여기서 쓰임! |
| 에피소드 경계 | `episodes.jsonl` | 에피소드별 길이 |
| 행동/상태 통계 | `stats.json` | 변환 스크립트가 계산 |

> 즉 변환 스크립트 하나가 "h5 -> (parquet + mp4 + meta json들)"로 펼치는 일을 합니다.
> 우리가 앞서 넣어둔 `label_metadata`(target_id->색)가 **언어 지시문 자동 생성**에 그대로 쓰여요.

## 9. 파인튜닝 실행 흐름 (당신의 `SFTTrainer` 자리)

```bash
# 1) 모델 받기 (HF) — 당신이 늘 하던 그 단계
huggingface-cli download nvidia/GR00T-N1.7-3B

# 2) 데이터 준비 — 위 8절의 변환으로 LeRobot 폴더 만들기 (+ modality.json)

# 3) 파인튜닝 — trl SFTTrainer 대신 GR00T 런처
python gr00t/experiment/launch_finetune.py \
    --base-model-path   nvidia/GR00T-N1.7-3B \
    --dataset-path      ./threecoloredcubes_lerobot \
    --embodiment-tag    NEW_EMBODIMENT \
    --modality-config-path ./configs/threecoloredcubes_config.py \
    --num-gpus 1 --global-batch-size 32 --action-horizon 16 --max-steps 2000
```

권고(레포 문서): 배치 최대화, 수천 step, 이미지 augmentation 때문에 run 간 5~6% 분산은 정상.
GPU는 ~20-24GB부터(3B급) — 당신의 클라우드 GPU면 충분.

## 10. 정리 — LLM SFT 경험을 어디에 그대로 쓰고, 어디만 새로 배우나

**그대로 쓰는 것 (친숙):**
- HF에서 base 모델 다운로드
- "데이터를 모델 형식으로 변환 -> 런처로 파인튜닝" 이라는 파이프라인 골격
- 정규화/배치/step/체크포인트 같은 학습 감각

**새로 배우는 것 (VLA 고유):**
1. 데이터가 **chat JSONL이 아니라 LeRobot 폴더**(parquet+mp4+meta)
2. **`modality.json` + embodiment tag** 가 "템플릿" 역할 — 어느 컬럼이 상태/행동/카메라인지
3. 정답이 **토큰이 아니라 행동 청크**, 손실이 **디퓨전 denoising**
4. **action horizon(청킹)** 과 **행동 공간(관절 vs EE 델타)** 결정
5. **proprioception(상태)** 을 명시적으로 입력/정규화

**다음에 정할 것 한 가지:** 행동 공간을 **관절 그대로(NEW_EMBODIMENT)** 갈지, **EE 델타로 변환**해 N1.7 관례에 맞출지.
이걸 정하면 `modality.json`과 변환 스크립트가 확정됩니다.

## 11. 자주 헷갈리는 점 (Q&A)

**Q1. Embodiment ID = ManiSkill task별 로봇인가?**
아니오. embodiment는 **로봇의 몸 + 상태/행동 규약**이지 task가 아닙니다.
ThreeColoredCubes / PickCube / StackCube는 전부 같은 Franka Panda + `pd_joint_pos` -> **embodiment 하나로 공유**.
새 embodiment를 정의할 때 = 로봇이 바뀌거나 행동공간(관절<->EE)이 바뀔 때만.
우리 하네스 전체(모든 Panda task, 관절 제어) = `NEW_EMBODIMENT` **한 개로 충분**.

**Q2. 영상(mp4)을 쓰니 `h5_add_images`는 건너뛰어도 되나?**
아니오. mp4는 **스텝별 프레임을 압축 저장**한 형식일 뿐, 모델은 여전히 스텝마다 한 프레임씩 먹습니다.
그 프레임을 *생성*하는 게 `h5_add_images`(리플레이 -> 렌더 -> rgb 기록). 원본 `motionplanning.h5`엔 이미지가 없습니다.
-> **`h5_add_images`는 필수.** 변환 단계는 그 rgb 프레임들을 **이어붙여 mp4로 인코딩**할 뿐.

```
task_to_h5 -> motionplanning.h5 (이미지 X)
  -> h5_add_images -> rgb 프레임 (T,128,128,3)        # 필수: 픽셀을 여기서 만든다
     -> [변환] 프레임 -> mp4 인코딩 + parquet/meta 작성 -> LeRobot 폴더
```

**Q3. row의 `task_index`가 Embodiment ID와 같은 건가?**
다릅니다. `task_index` = `tasks.jsonl`의 **언어 지시문**(빨강/초록/파랑 -> 에피소드마다 변함).
Embodiment ID = **어떤 로봇/규약**(Panda 고정). 한 데이터셋에 task_index는 여러 개, embodiment는 보통 하나.
요약: **task_index = 무엇을 하라(언어)**, **embodiment = 어떤 몸으로**.

## 12. 한 샘플에 대한 1-스텝 학습 (당신의 SFTTrainer를 손으로 푼 버전)

> 실행용 아님 — Isaac-GR00T API 패턴을 따른 설명용. 정확한 클래스/필드명은 설치 버전으로 확인.
> 익숙한 부분(`from_pretrained`, optimizer, `backward`, `step`)은 **그대로**,
> 다른 건 딱 둘: (1) 입력이 토큰 시퀀스가 아니라 **video+state+language 딕셔너리**,
> (2) `loss`가 next-token CE가 아니라 **행동 청크의 디퓨전/플로우 손실**.

In [ ]:
import torch
from gr00t.data.dataset import LeRobotSingleDataset
from gr00t.data.embodiment_tags import EmbodimentTag
from gr00t.experiment.data_config import DATA_CONFIG_MAP
from gr00t.model.gr00t_n1 import GR00T_N1            # 정책/모델 클래스 (버전마다 이름 다를 수 있음)

# (1) data config = 우리 embodiment의 'modality.json + transforms' 묶음
data_config     = DATA_CONFIG_MAP["custom_panda"]    # 우리가 등록할 설정
modality_config = data_config.modality_config()
transforms      = data_config.transform()

# (2) 데이터셋: 우리 LeRobot 폴더를 가리킴
dataset = LeRobotSingleDataset(
    dataset_path     = "./threecoloredcubes_lerobot",
    modality_configs = modality_config,
    transforms       = transforms,
    embodiment_tag   = EmbodimentTag.NEW_EMBODIMENT,
)

# (3) 샘플 1개 = 당신의 '토큰화된 chat 한 건'에 대응 (입력 video/state/language + 정답 action 청크)
sample = dataset[0]
batch  = {k: (v.unsqueeze(0) if torch.is_tensor(v) else [v]) for k, v in sample.items()}  # batch=1

# (4) 모델 로드 (당신이 늘 하던 from_pretrained)
model = GR00T_N1.from_pretrained("nvidia/GR00T-N1.7-3B").cuda().train()
batch = {k: (v.cuda() if torch.is_tensor(v) else v) for k, v in batch.items()}

# (5) 1 스텝 학습 — LLM과 골격 동일, 단 loss가 CE가 아니라 디퓨전/플로우
opt  = torch.optim.AdamW(model.parameters(), lr=1e-4)
out  = model(batch)        # forward: action 청크에 입힌 노이즈를 예측
loss = out.loss            # action 청크에 대한 디퓨전 denoising 손실 (CE 아님)
loss.backward()
opt.step(); opt.zero_grad()
print("loss:", loss.item())

# 실제로는 이 루프를 직접 안 쓰고 launch_finetune.py(HF Trainer 기반)가 (2)~(5)를 감싼다
# — SFTTrainer가 루프를 감춰주는 것과 같다.